# RGB↔X: Image Decomposition and Synthesis

This notebook runs the RGB→X and X→RGB models on Google Colab with GPU support.

**Make sure to enable GPU:** Runtime → Change runtime type → GPU (T4, V100, or A100)

## Setup: Clone Repository and Install Dependencies

**First-time setup:** You'll need a GitHub Personal Access Token to clone the private repo.

### How to generate token:
1. Go to: https://github.com/settings/tokens
2. Click **"Generate new token (classic)"**
3. Name it: `Colab RGB-X`
4. Check: **`repo`** (Full control of private repositories)
5. Click **"Generate token"** at bottom
6. **Copy the token** (you won't see it again!)

Paste the token when prompted below ⬇️

In [ ]:
# Clone YOUR private repo (includes shadow download functionality)
# This requires a Personal Access Token for private repo access

from getpass import getpass
import os

print("🔐 GitHub Authentication Required")
print("Generate token at: https://github.com/settings/tokens")
print("Permissions needed: 'repo' (full control)")
print()

token = getpass('Enter your GitHub Personal Access Token: ')

# Clone using token
!git clone https://{token}@github.com/id-21/rgb-x.git

# Clear token from memory for security
token = None

%cd rgb-x/rgbx

In [ ]:
# Force Python 3.9 environment (matching original environment.yml)
# This is CRITICAL - the code was built for Python 3.9, Colab defaults to 3.12
!apt-get update -qq
!apt-get install -y python3.9 python3.9-venv python3.9-dev

# Create and activate Python 3.9 virtual environment
!python3.9 -m venv /content/py39env
!source /content/py39env/bin/activate && pip install --upgrade pip

# Install PyTorch with CUDA 12.1 support (in Python 3.9 environment)
!source /content/py39env/bin/activate && pip install torch==2.3.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install exact versions from environment.yml
!source /content/py39env/bin/activate && pip install diffusers==0.20.0 gradio==3.48.0 imageio==2.34.1 numpy==1.26.4 opencv-python==4.9.0.80 transformers==4.40.2 huggingface-hub==0.25.0

print("\\n✓ Python 3.9 environment created with all dependencies")

In [ ]:
# Verify GPU is available (using Python 3.9 environment)
!/content/py39env/bin/python -c "import torch; print(f'CUDA available: {torch.cuda.is_available()}'); print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"None\"}')"

## Run Demos

Run the cells below to launch the Gradio demos. A public URL will be generated that you can access in your browser.

### Patch Gradio Demos for Colab

This cell fixes two Colab-specific issues:
1. Clears matplotlib backend environment variable (Colab sets this for Jupyter, conflicts with standalone Python)
2. Enables public URL sharing for Gradio (localhost not accessible in Colab)

In [ ]:
import os

# Fix 1: Clear matplotlib backend issue (Colab sets this for Jupyter, conflicts with standalone Python)
os.environ.pop('MPLBACKEND', None)

# Fix 2: Patch gradio demos to enable public URL sharing
def patch_gradio_demo(file_path):
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Replace demo.launch() with demo.launch(share=True)
    content = content.replace('demo.launch()', 'demo.launch(share=True)')
    
    with open(file_path, 'w') as f:
        f.write(content)
    
    print(f"✓ Patched {file_path}")

# Patch both demos
patch_gradio_demo('/content/rgb-x/rgbx/rgb2x/gradio_demo_rgb2x.py')
patch_gradio_demo('/content/rgb-x/rgbx/x2rgb/gradio_demo_x2rgb.py')

print("\n✓ All fixes applied. Ready to launch demos!")
print("📥 RGB→X demo now includes shadow download buttons for ST-RoomNet V3!")

In [ ]:
### Launch RGB→X Demo

**Download the "Shadow Layer (EXR)" file to use in ST-RoomNet V3!**

This will generate a public Gradio URL (https://xxx.gradio.live) that you can access in your browser.

In [ ]:
# Launch RGB→X Gradio demo (using Python 3.9)
%cd rgb2x
!/content/py39env/bin/python gradio_demo_rgb2x.py

In [ ]:
### Launch X→RGB Demo

**Note:** Stop the RGB→X demo above (interrupt kernel) before running this cell.

This will generate a public Gradio URL (https://xxx.gradio.live) that you can access in your browser.

## X→RGB: Synthesize Images from Intrinsic Channels

**Note:** Stop the RGB→X demo above (interrupt kernel) before running this cell.

In [ ]:
# Launch X→RGB Gradio demo (using Python 3.9)
%cd ../x2rgb
!/content/py39env/bin/python gradio_demo_x2rgb.py

## Alternative: Run Inference Programmatically

If you want to run inference without Gradio, you can load the models directly:

In [ ]:
# Example: Load and use RGB→X model programmatically
import sys
sys.path.append('/content/rgbx/rgb2x')

from PIL import Image
from diffusers import StableDiffusionPipeline
import torch

# Load your image
# input_image = Image.open('path/to/your/image.jpg')

# Model loading code will depend on the actual implementation in gradio_demo_rgb2x.py
# You may need to inspect that file to see how models are loaded

## Upload Your Own Images

Use the file upload widget below to upload images for testing:

In [ ]:
from google.colab import files
uploaded = files.upload()

# Your uploaded files will be in the current directory
for filename in uploaded.keys():
    print(f'Uploaded: {filename}')